# Sistema de recomendação de filmes por similaridade de sinopses

**Projeto individual · Trilha A — Sistemas de Recomendação**

Este notebook reúne a documentação, o código e os resultados do projeto usando a base `movies_metadata.csv`. Execute as células de cima para baixo. Os resultados salvos foram calculados com o arquivo fornecido.

## 1. O problema

O objetivo é sugerir até dez filmes a partir de um título que o usuário conhece. A hipótese é que palavras das sinopses ajudam a identificar personagens, cenários e temas relacionados. A abordagem escolhida é a recomendação baseada em conteúdo, adequada aos metadados disponíveis. A interação consiste em selecionar um filme e receber uma lista ordenada por similaridade textual.

## 2. O dataset

A base contém **45.466 registros e 24 colunas**, com título, sinopse, gêneros, data de lançamento, coleção, média de votos e quantidade de votos, entre outros campos. Cada registro representa um filme. O arquivo integra *The Movies Dataset*, de Rounak Banik, que reúne metadados do TMDB e arquivos de avaliações do MovieLens. Neste projeto foi utilizado o arquivo de metadados fornecido. [The Movies Dataset — Kaggle](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)

| Campo | Uso no projeto |
|---|---|
| `id` | Identificação e remoção de duplicatas |
| `title`, `release_date` | Apresentação e identificação dos resultados |
| `overview` | Construção dos vetores de texto |
| `belongs_to_collection` | Referência para avaliação das recomendações |
| `vote_count` | Comparação com um ranking dos filmes mais votados |

### Como executar

No **Google Colab**, abra este arquivo e execute as células. Quando solicitado, envie `movies_metadata.csv` pelo botão de upload. No **Jupyter**, mantenha o CSV na pasta de trabalho do notebook ou ajuste `CAMINHO_CSV` na célula de carregamento.

O código usa Python 3.10 ou superior e NumPy. O Colab já inclui NumPy. Se precisar instalar em outro ambiente, execute `%pip install numpy` em uma célula e reinicie o kernel.


In [1]:
import ast
from collections import Counter, defaultdict
import csv
from dataclasses import dataclass
import hashlib
from math import log, sqrt
from pathlib import Path
import platform
import re

import numpy as np

print(f"Python {platform.python_version()} | NumPy {np.__version__}")


Python 3.12.14 | NumPy 2.3.5


In [2]:
# Ajuste este caminho se o CSV estiver em outra pasta.
CAMINHO_CSV = Path("movies_metadata.csv")

if not CAMINHO_CSV.is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            "Coloque movies_metadata.csv na pasta de trabalho ou ajuste CAMINHO_CSV."
        ) from None
    print("Selecione o arquivo movies_metadata.csv.")
    files.upload()

if not CAMINHO_CSV.is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {CAMINHO_CSV}")

with CAMINHO_CSV.open("rb") as arquivo:
    checksum = hashlib.md5(arquivo.read()).hexdigest()
with CAMINHO_CSV.open(encoding="utf-8-sig", newline="") as arquivo:
    colunas = next(csv.reader(arquivo))

print(f"Arquivo: {CAMINHO_CSV.name}")
print(f"Colunas: {len(colunas)}")
print(f"MD5: {checksum}")


Arquivo: movies_metadata.csv
Colunas: 24
MD5: 42bf3ef8c208a01a4776955875978b1e


## 3. Tratamento e limpeza

A leitura respeita as aspas e quebras de linha do CSV. O tratamento valida identificadores, elimina registros sem título ou sem sinopse utilizável e reconhece marcadores de conteúdo ausente, como *No overview found*. Duplicatas são identificadas pelo ID do TMDB. Entre registros válidos do mesmo ID, prevalece a sinopse mais longa. Empates preservam a primeira ocorrência.

As exclusões são contadas na ordem em que ocorrem. A estrutura da coleção é lida com `ast.literal_eval`. Cada filme é armazenado com os campos necessários ao recomendador e à avaliação.


In [3]:
TOKEN = re.compile(r"[^\W\d_]{2,}")
SINOPSES_AUSENTES = {
    "no movie overview available",
    "no movie overview available, please add one at themoviedb.org",
    "no overview",
    "no overview found",
    "no overview yet",
    "no plot overview available",
    "not available",
}
K = 10


@dataclass
class Filme:
    id: int
    titulo: str
    ano: str
    sinopse: str
    votos: int
    colecao: int | None

def carregar_catalogo(caminho):
    filmes = {}
    contagens = Counter()
    with open(caminho, encoding="utf-8-sig", newline="") as arquivo:
        for linha in csv.DictReader(arquivo):
            contagens["linhas lidas"] += 1
            identificador = (linha["id"] or "").strip()
            if not re.fullmatch(r"[1-9][0-9]*", identificador):
                contagens["ID inválido"] += 1
                continue
            titulo = (linha["title"] or "").strip()
            if not titulo:
                contagens["sem título"] += 1
                continue
            sinopse = " ".join((linha["overview"] or "").split())
            if not TOKEN.search(sinopse):
                contagens["sem sinopse utilizável"] += 1
                continue
            if sinopse.casefold().rstrip(".") in SINOPSES_AUSENTES:
                contagens["marcador de sinopse ausente"] += 1
                continue

            id_filme = int(identificador)
            anterior = filmes.get(id_filme)
            if anterior is not None:
                contagens["linhas repetidas por ID"] += 1
                if len(anterior.sinopse) >= len(sinopse):
                    continue

            votos = (linha["vote_count"] or "").strip()
            colecao = (linha["belongs_to_collection"] or "").strip()
            filmes[id_filme] = Filme(
                id_filme,
                titulo,
                (linha["release_date"] or "")[:4],
                sinopse,
                int(votos) if votos.isdecimal() else 0,
                ast.literal_eval(colecao)["id"] if colecao else None,
            )
    return sorted(filmes.values(), key=lambda filme: filme.id), contagens


In [4]:
filmes, contagens = carregar_catalogo(CAMINHO_CSV)
if not filmes:
    raise ValueError("A base não contém filmes com sinopse utilizável.")

print(f"Registros originais: {contagens['linhas lidas']:,}")
for motivo in [
    "ID inválido",
    "sem título",
    "sem sinopse utilizável",
    "marcador de sinopse ausente",
    "linhas repetidas por ID",
]:
    print(f"Exclusões por {motivo}: {contagens[motivo]:,}")
print(f"Catálogo final: {len(filmes):,} filmes")


Registros originais: 45,466
Exclusões por ID inválido: 3
Exclusões por sem título: 3
Exclusões por sem sinopse utilizável: 961
Exclusões por marcador de sinopse ausente: 152
Exclusões por linhas repetidas por ID: 30
Catálogo final: 44,317 filmes


## 4. Metodologia

### Representação TF-IDF

As sinopses são convertidas para minúsculas e divididas em sequências de pelo menos duas letras. São mantidos os termos presentes em pelo menos dois filmes e em até 20% do catálogo. O vocabulário final é calculado durante a execução.

O peso combina a frequência do termo na sinopse e a frequência inversa de documentos, com suavização.

$$w_{t,d} = tf_{t,d}\left[\ln\left(\frac{1+N}{1+df_t}\right)+1\right]$$

Aqui, $N$ é a quantidade de filmes e $df_t$ é a quantidade de sinopses que contêm o termo. Os vetores são normalizados para comprimento unitário. A filtragem por frequência reduz termos muito comuns. A representação preserva as formas das palavras, sem lematização. [TF-IDF — documentação do scikit-learn](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)

### Similaridade do cosseno

O sistema compara o vetor do filme consultado aos vetores do catálogo e ordena os resultados pela similaridade.

$$\operatorname{sim}(a,b)=\frac{a\cdot b}{\|a\|_2\|b\|_2}$$

O próprio filme é excluído da lista. Os empates são resolvidos pelo ID crescente. São retornados até dez candidatos com similaridade positiva. Um índice invertido permite acumular as contribuições das palavras compartilhadas sem armazenar uma matriz densa de todos os pares de filmes. A implementação utiliza Python e NumPy. [Similaridade do cosseno](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)


In [5]:
class IndiceTfidf:
    def __init__(self, filmes):
        documentos = [Counter(TOKEN.findall(filme.sinopse.lower())) for filme in filmes]
        frequencia = Counter(termo for doc in documentos for termo in doc)
        total = len(filmes)
        idf = {
            termo: log((1 + total) / (1 + ocorrencias)) + 1
            for termo, ocorrencias in frequencia.items()
            if 2 <= ocorrencias <= 0.20 * total
        }
        self.vetores = []
        ocorrencias_por_termo = defaultdict(list)
        for posicao, doc in enumerate(documentos):
            pesos = {termo: qtd * idf[termo] for termo, qtd in doc.items() if termo in idf}
            norma = sqrt(sum(peso ** 2 for peso in pesos.values()))
            vetor = {termo: peso / norma for termo, peso in pesos.items()}
            self.vetores.append(vetor)
            for termo, peso in vetor.items():
                ocorrencias_por_termo[termo].append((posicao, peso))

        self.ocorrencias = {}
        for termo, lista in ocorrencias_por_termo.items():
            posicoes, pesos = zip(*lista)
            self.ocorrencias[termo] = (np.asarray(posicoes, dtype=np.intp), np.asarray(pesos))

    def recomendar(self, consulta, k=K):
        similaridades = np.zeros(len(self.vetores))
        for termo, peso in self.vetores[consulta].items():
            posicoes, pesos = self.ocorrencias[termo]
            similaridades[posicoes] += peso * pesos
        similaridades[consulta] = 0

        candidatos = np.flatnonzero(similaridades > 0)
        ordem = np.lexsort((candidatos, -similaridades[candidatos]))[:k]
        return [(int(i), float(similaridades[i])) for i in candidatos[ordem]]


In [6]:
indice = IndiceTfidf(filmes)
print(f"Filmes indexados: {len(indice.vetores):,}")
print(f"Vocabulário: {len(indice.ocorrencias):,} termos")
print(f"Filmes sem termos no vocabulário: {sum(not v for v in indice.vetores)}")


Filmes indexados: 44,317
Vocabulário: 37,419 termos
Filmes sem termos no vocabulário: 0


## 5. Geração das recomendações

A consulta aceita um título exato ou o ID do TMDB. Títulos repetidos exibem opções com ano e ID para que a seleção seja precisa. O escore exibido é a similaridade entre as sinopses, em escala de zero a um.


In [7]:
def mostrar_recomendacoes(filmes, indice, consulta):
    filme = filmes[consulta]
    print(f"\n{filme.titulo} ({filme.ano or 'ano desconhecido'}) | ID {filme.id}")
    recomendacoes = indice.recomendar(consulta)
    if not recomendacoes:
        print("Sem filmes com similaridade positiva.")
    for posicao, (i, escore) in enumerate(recomendacoes, start=1):
        filme = filmes[i]
        print(f"{posicao:2}. {filme.titulo} ({filme.ano or 'ano desconhecido'}) | ID {filme.id} | {escore:.4f}")


def recomendar_filme(identificador):
    termo = str(identificador).strip().casefold()
    consultas = [i for i, filme in enumerate(filmes) if str(filme.id) == termo]
    if not consultas:
        consultas = [i for i, filme in enumerate(filmes) if filme.titulo.casefold() == termo]
    if not consultas:
        raise ValueError(f"Filme não encontrado: {identificador}")
    if len(consultas) > 1:
        opcoes = ", ".join(
            f"{filmes[i].titulo} ({filmes[i].ano}) ID {filmes[i].id}" for i in consultas
        )
        raise ValueError(f"Título ambíguo. Consulte pelo ID: {opcoes}")
    mostrar_recomendacoes(filmes, indice, consultas[0])


In [8]:
# Troque por outro título exato ou ID do TMDB, por exemplo 603.
FILME = "Toy Story"
recomendar_filme(FILME)



Toy Story (1995) | ID 862
 1. Toy Story 3 (2010) | ID 10193 | 0.4991
 2. Toy Story 2 (1999) | ID 863 | 0.4683
 3. The 40 Year Old Virgin (2005) | ID 6957 | 0.2854
 4. Small Fry (2011) | ID 82424 | 0.2662
 5. Andy Hardy's Blonde Trouble (1944) | ID 92848 | 0.2255
 6. Superstar: The Life and Times of Andy Warhol (1990) | ID 250434 | 0.2116
 7. Andy Kaufman Plays Carnegie Hall (1980) | ID 364123 | 0.2116
 8. Hot Splash (1988) | ID 181801 | 0.2011
 9. Andy Peters: Exclamation Mark Question Point (2015) | ID 355984 | 0.1948
10. The Champ (1931) | ID 42816 | 0.1934


In [9]:
for titulo in ["Finding Nemo", "The Matrix"]:
    recomendar_filme(titulo)



Finding Nemo (2003) | ID 12
 1. Finding Dory (2016) | ID 127380 | 0.2850
 2. The Reef (2010) | ID 49787 | 0.1863
 3. Mr. Nobody (2009) | ID 31011 | 0.1686
 4. Captain Nemo and the Underwater City (1969) | ID 56354 | 0.1608
 5. 20,000 Leagues Under the Sea (1954) | ID 173 | 0.1495
 6. The Reef 2: High Tide (2012) | ID 132601 | 0.1356
 7. 20,000 Leagues Under the Sea (1997) | ID 2965 | 0.1203
 8. Under the Boardwalk (1989) | ID 89909 | 0.1162
 9. Pororo: The Racing Adventure (2013) | ID 226673 | 0.1128
10. Under the Sea 3D (2009) | ID 36123 | 0.1118

The Matrix (1999) | ID 603
 1. A Detective Story (2003) | ID 24660 | 0.2393
 2. Kid's Story (2003) | ID 24914 | 0.2079
 3. World Record (2003) | ID 24960 | 0.1989
 4. Algorithm (2014) | ID 281826 | 0.1923
 5. Hackers (1995) | ID 10428 | 0.1846
 6. Who Am I (2014) | ID 284427 | 0.1747
 7. Berlin Express (1948) | ID 42286 | 0.1739
 8. Underground: The Julian Assange Story (2012) | ID 136698 | 0.1735
 9. Commando (1985) | ID 10999 | 0.1724
10.

## 6. Avaliação dos resultados

O teste usa `belongs_to_collection` como referência aproximada de relevância. Para cada filme de uma coleção com pelo menos dois títulos válidos, os outros integrantes dessa coleção são considerados relevantes. Todos os filmes do catálogo tratado podem concorrer às dez posições. O identificador da coleção participa somente da avaliação. A referência de comparação recomenda os filmes com maior `vote_count`, excluindo a consulta.

O protocolo avalia a recuperação dentro de um catálogo fixo. O IDF é calculado sobre esse catálogo. A generalização para novos filmes e as preferências individuais exigiriam experimentos adicionais. A base disponível permite medir relações entre itens por coleção. Filmes de outras coleções podem ser pertinentes e ficam fora da contagem de relevantes deste teste.

As métricas são calculadas por consulta e depois promediadas.

| Métrica | Definição para as dez primeiras posições |
|---|---|
| Hit@10 | Fração das consultas com pelo menos um filme relevante |
| Precisão@10 | Número de filmes relevantes recuperados dividido por 10 |
| Recall@10 | Número de filmes relevantes recuperados dividido pelo total de relevantes da consulta |
| MRR@10 | Inverso da posição do primeiro relevante, com zero quando nenhum aparece |

Execute a próxima célula para avaliar todas as coleções elegíveis. O processamento pode levar alguns minutos, conforme o ambiente.


In [10]:
def avaliar(filmes, indice):
    colecoes = defaultdict(set)
    for posicao, filme in enumerate(filmes):
        if filme.colecao is not None:
            colecoes[filme.colecao].add(posicao)

    popularidade = sorted(range(len(filmes)), key=lambda i: (-filmes[i].votos, filmes[i].id))
    somas = {"TF-IDF + cosseno": Counter(), "Mais votados": Counter()}
    consultas = 0
    colecoes_avaliadas = 0
    for grupo in colecoes.values():
        if len(grupo) < 2:
            continue
        colecoes_avaliadas += 1
        for consulta in sorted(grupo):
            consultas += 1
            relevantes = grupo - {consulta}
            ranking = {
                "TF-IDF + cosseno": [i for i, _ in indice.recomendar(consulta)],
                "Mais votados": [i for i in popularidade[: K + 1] if i != consulta][:K],
            }
            for metodo, lista in ranking.items():
                posicoes = [pos for pos, i in enumerate(lista, start=1) if i in relevantes]
                somas[metodo]["hit"] += bool(posicoes)
                somas[metodo]["precisao"] += len(posicoes) / K
                somas[metodo]["recall"] += len(posicoes) / len(relevantes)
                somas[metodo]["mrr"] += 1 / posicoes[0] if posicoes else 0

    print(f"\nAvaliação top {K} | {consultas} consultas de {colecoes_avaliadas} coleções")
    if not consultas:
        return
    for metodo, medidas in somas.items():
        print(
            f"{metodo}: {int(medidas['hit'])} consultas com pelo menos um relevante; "
            f"Hit@{K}={medidas['hit'] / consultas:.4%}; "
            f"Precisão@{K}={medidas['precisao'] / consultas:.4%}; "
            f"Recall@{K}={medidas['recall'] / consultas:.4%}; "
            f"MRR@{K}={medidas['mrr'] / consultas:.5f}"
        )


In [11]:
avaliar(filmes, indice)



Avaliação top 10 | 4030 consultas de 1287 coleções
TF-IDF + cosseno: 2489 consultas com pelo menos um relevante; Hit@10=61.7618%; Precisão@10=13.7072%; Recall@10=43.7173%; MRR@10=0.46467
Mais votados: 8 consultas com pelo menos um relevante; Hit@10=0.1985%; Precisão@10=0.0199%; Recall@10=0.1241%; MRR@10=0.00050


### Resultados obtidos com a base fornecida

A limpeza resultou em **44.317 filmes** e o vocabulário teve **37.419 termos**. A avaliação cobriu **4.030 consultas de 1.287 coleções**. O TF-IDF recuperou pelo menos um integrante da coleção em **2.489 consultas**. A lista de mais votados alcançou esse resultado em **8 consultas**.

| Métrica | TF-IDF + cosseno | Mais votados |
|---|---:|---:|
| Hit@10 | 61,76% | 0,20% |
| Precisão@10 | 13,71% | 0,02% |
| Recall@10 | 43,72% | 0,12% |
| MRR@10 | 0,46467 | 0,00050 |

Os valores desta tabela registram a execução sobre o arquivo fornecido, com MD5 `42bf3ef8c208a01a4776955875978b1e`. Ao substituir a base, compare os resultados das células executadas com esse registro.

### Análise dos exemplos e limitações

Para *Toy Story*, as duas primeiras sugestões foram *Toy Story 3* e *Toy Story 2*. Para *Finding Nemo*, a primeira foi *Finding Dory*. O teste também revelou uma limitação lexical. *The 40 Year Old Virgin* apareceu em terceiro para *Toy Story*, principalmente pela palavra **Andy**, usada para personagens diferentes nas duas sinopses.

Em *The Matrix*, a lista incluiu *A Detective Story*, *Kid's Story* e *World Record* nas primeiras posições. Os dois longas seguintes da coleção ficaram fora do top 10. Esse caso mostra que a recuperação de uma coleção depende do vocabulário efetivamente presente nas sinopses.

## 7. Conclusão e trabalhos futuros

A comparação por sinopses recuperou integrantes de coleções com maior frequência que o ranking dos filmes mais votados. Os exemplos indicam que termos ligados a personagens e cenários ajudam na recuperação, enquanto nomes ambíguos podem gerar sugestões pouco relacionadas.

Os próximos experimentos podem combinar sinopses e gêneros, testar representações semânticas e comparar os resultados com o mesmo protocolo. Uma avaliação com usuários permitirá examinar pertinência e satisfação. Com avaliações individuais e a correspondência correta dos identificadores, será possível comparar a filtragem colaborativa e uma abordagem híbrida. [MovieLens — GroupLens](https://grouplens.org/datasets/movielens/)
